# A Demo on Reinforcement Learning (based on the book Understanding Deep Learning by Simon J. D. Prince[[1]](#references))


## Overview
Reinforcement learning is a type of machine learning where an agent learns to make decisions by interacting with an environment, receiving rewards for good behaviour and penalties for bad behaviour.

Chess is therefore a great option for reinforcement learning because
- It has clear rules and goals,
- Every move affects future possibilities,
- It allows the agent to explore, learn from mistakes, and improve over time.


This is our notebook on Reinforcement Learning where we train two agents to play BulletChess.
We have two agents:
* a DDQN agent: This agent is based on a double-deep Q-Network model
* a Value-Policy agent: This agent uses MCTS (Monte-Carlo Tree-Search) for action selection and AlphaZeroNet for policy and value estimation.

The aim of this project is to develop reinforcement learning agents for bullet chess, in which each player has only 60 seconds in total to complete the game.
In this notebook you can either train the agents yourself or used saved models.
There is the option to play against either of our agents yourself, choosing which color you want to play.
Feel free to try around.

## Basic Initialisation

Before we start actually working on the models, we need to import all relevant modules and functions. 

In [1]:
from src.environment import BulletChessEnv
from src.training import BulletChessDDQNTrainer, BulletChessAlphaZeroTrainer
from src.utils import get_truly_fixed_cfg
import matplotlib as plt
import src.play

## Training the agents

### DDQN Agent

We started by implementing the DualingQNet class, which is a deep neural network used to estimate Q-values.
To handle experience, we added two buffer classes:
- NStepBuffer for multi-step transitions, and
- PERBuffer for prioritized experience replay.
Finally, everything then is tied together in the DQNAgent class, which handles training, action selection, and network updates.

The actual training is defined in the BulletChessDDQNTrainer class, which handles the full training pipeline. Using this class, we then define our trainer. All models and checkpoints will then be saved as ... in the subfolder .\models. 

For detailed implementation details, have a look at the respecting files in .\scr. 

In [ ]:
cfg = get_truly_fixed_cfg()
env = BulletChessEnv(
            time_limit=cfg["env"]["time_limit"],
            increment=cfg["env"]["increment"],
            simulate_think=cfg["env"]["simulate_think"],
            think_lo=cfg["env"]["think_lo"],
            think_hi=cfg["env"]["think_hi"]
        )
ddqn_trainer = BulletChessDDQNTrainer(cfg, env)

print("=== Environment Test ===")
state = env.reset()
print(f"Initial state shape: {state.shape}")
print(f"Initial board: {env.state.board}")
print(f"Legal moves: {len(env.get_legal_actions())}")
print(f"Game over: {env.is_game_over()}")

print("=== Starting DDQN Training ===")
ddqn_trainer.train("time") # "time" or "episodes" can be used to specify the training type

### Policy-Value Agent

We implemented MCTSAgent, a policy-value agent using an AlphaZero-style neural network (AlphaZeroNet) with separate policy and value heads. The agent uses Monte Carlo Tree Search (MCTS) guided by the network’s policy priors to select actions.

The AlphaZeroNet processes the board state and outputs policy logits and a scalar value.
MCTS then runs multiple simulations to improve the policy by exploring actions with a UCB score.

The agent runs MCTS to generate improved action probabilities, selects actions by sampling from this policy, trains the network on self-play data using combined policy and value losses, and supports saving and loading checkpoints.

This agent works with the BulletChessEnv environment as well and relies solely on MCTS for exploration (no epsilon-greedy or target networks).

In [ ]:
cfg = get_truly_fixed_cfg()
env = BulletChessEnv(
            time_limit=cfg["env"]["time_limit"],
            increment=cfg["env"]["increment"],
            simulate_think=cfg["env"]["simulate_think"],
            think_lo=cfg["env"]["think_lo"],
            think_hi=cfg["env"]["think_hi"]
        )
alpha_trainer = BulletChessAlphaZeroTrainer(cfg, env)

print("=== Environment Test ===")
state = env.reset()
print(f"Initial state shape: {state.shape}")
print(f"Initial board: {env.state.board}")
print(f"Legal moves: {len(env.get_legal_actions())}")
print(f"Game over: {env.is_game_over()}")

print("=== Starting AlphaZero Training ===")
alpha_trainer.train("time") # "time" or "episodes" can be used to specify the training type

## Playing Chess with the Agents

### Starting the Game
1. Select a model from the dropdown list — this is the AI opponent.
2. Choose your color: White or Black.
3. Click "Start Game".

### Making Moves
- Type your move into the ""Move:" input box.
- You can enter moves in **standard algebraic notation** (e.g., `Nf3`, `e4`) or **UCI notation** (e.g., `e2e4`).
- The input box offers autochecks for all legal moves.
- Press Enter to submit your move.

### Game Flow
- After you play your move, the AI will think and make its move automatically.
- The board and game info (move count, time, last move) update live.
- You cannot play while the AI is thinking.

### Controls
- **Restart**: Starts a new game with the same settings.
- **Replay**: Replays the moves from the current game one by one.

### Notes
- If you enter an invalid move, you’ll get a warning and can try again.
- The AI uses the selected model to decide its moves.
- The game ends when checkmate, stalemate, or draw occurs.
- Technically, you can open multiple boards at once. But we recommend only starting one game (might take a few seconds, be patient!).

Enjoy your game!

In [2]:
launcher = src.play.JupyterChessLauncher()
display(launcher.ui)